# Devnagri LLM — Clean Kaggle Pipeline (Hindi Only)

In [1]:
from pathlib import Path
for p in Path("/kaggle/input").iterdir():
    print(p)
    for sub in p.rglob("persistent"):
        print("  found:", sub)

/kaggle/input/notebooks
  found: /kaggle/input/notebooks/parthbramhecha77/devnagri-restarte/persistent
/kaggle/input/datasets


In [2]:
# ============================================================
# 0. KAGGLE / RUNTIME CONFIGURATION
# ============================================================
import os
import sys
import time
from pathlib import Path

os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")

REPO_DIR = Path("/kaggle/working/Devnagri_LLM")
PERSISTENT_DIR = Path("/kaggle/working/persistent")
SPLIT_SOURCE = Path(
    "/kaggle/input/datasets/parthbramhecha77/"
    "split-devnagri-compression/splits"
)

# Hindi-only run. (Full pipeline normally loops over
# pipeline.config.LANGUAGES == ["hindi", "marathi", "sanskrit"];
# this notebook restricts every stage below to hindi.)
LANGUAGES = ["hindi"]

os.chdir("/kaggle/working")
PERSISTENT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Session wall-clock budget.
#
# Kaggle kills the *whole kernel* at a hard 12h GPU-session limit --
# not a catchable Python exception, just a SIGKILL. Every later stage
# in this notebook (Stage 2d's resumable rounds, the Stage 2d guard,
# Stage 3) must measure itself against THIS clock, not against its
# own internal --max-hours budget, or a stage can start with (say)
# 20 minutes of real time left and get killed mid-run with no
# checkpoint, no traceback, and no results file.
# ------------------------------------------------------------------
SESSION_START = time.time()
KAGGLE_SESSION_LIMIT_HOURS = 12.0   # Kaggle's hard wall-clock cap for this session
SESSION_SAFETY_BUFFER_HOURS = 1.5   # reserved for Stage 3 model load + 3-condition compression + Stage 4

def hours_elapsed():
    """Wall-clock hours since this notebook/session started."""
    return (time.time() - SESSION_START) / 3600.0

def hours_remaining_in_session():
    """Hours left before Kaggle's hard 12h cap, minus the safety buffer."""
    return KAGGLE_SESSION_LIMIT_HOURS - SESSION_SAFETY_BUFFER_HOURS - hours_elapsed()

print("=" * 72)
print("DEVNAGRI LLM — CLEAN KAGGLE PIPELINE (HINDI ONLY)")
print("=" * 72)
print("Python:", sys.version)
print("Working directory:", Path.cwd())
print("Repository:", REPO_DIR)
print("Dataset source:", SPLIT_SOURCE)
print("Languages:", ", ".join(LANGUAGES))
print(f"Session budget: {KAGGLE_SESSION_LIMIT_HOURS:.1f}h hard cap, "
      f"{SESSION_SAFETY_BUFFER_HOURS:.1f}h reserved for Stage 3/4 "
      f"-> {hours_remaining_in_session():.2f}h available for Stage 2d rounds.")


DEVNAGRI LLM — CLEAN KAGGLE PIPELINE (HINDI ONLY)
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Working directory: /kaggle/working
Repository: /kaggle/working/Devnagri_LLM
Dataset source: /kaggle/input/datasets/parthbramhecha77/split-devnagri-compression/splits
Languages: hindi
Session budget: 12.0h hard cap, 1.5h reserved for Stage 3/4 -> 10.50h available for Stage 2d rounds.


In [3]:
# ============================================================
# 0b. RESTORE CHECKPOINT STATE FROM A PREVIOUS SESSION (IF ATTACHED)
# ============================================================
import shutil
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

def find_prior_persistent_dir():
    """Locate a 'persistent' directory inside any attached input dataset.

    This is how a PREVIOUS committed version of THIS SAME notebook's Output
    gets found, once it is attached as an input to the current version:
    Notebook -> Add Input -> Notebook Output Files -> this notebook,
    latest version. Kaggle mounts it read-only under /kaggle/input/...,
    but the exact depth varies by input type:
      - a flat "Dataset" input mounts at    /kaggle/input/<slug>/...
      - a "Notebook Output Files" input mounts nested under
        /kaggle/input/notebooks/<username>/<notebook-slug>/...
    so we search recursively (rglob) rather than assuming a fixed depth,
    and pick the shallowest match in case more than one 'persistent'
    directory happens to be attached.
    """
    if not INPUT_ROOT.exists():
        return None
    matches = [p for p in INPUT_ROOT.rglob("persistent") if p.is_dir()]
    if not matches:
        return None
    matches.sort(key=lambda p: len(p.parts))
    return matches[0]

prior_persistent = find_prior_persistent_dir()

if prior_persistent is not None:
    print(f"↺ Found checkpoint state from a previous session: {prior_persistent}")
    PERSISTENT_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copytree(prior_persistent, PERSISTENT_DIR, dirs_exist_ok=True)

    restored_finals = sorted(
        p.parent.name for p in PERSISTENT_DIR.glob("models/devaware_finetuned/*/final")
    )
    restored_steps = sorted(
        f"{p.parent.name}/{p.name}"
        for p in PERSISTENT_DIR.glob("models/devaware_finetuned/*/step_*")
    )
    restored_results = sorted(
        p.parent.name for p in PERSISTENT_DIR.glob("results/*/compression_results.json")
    )
    print(f"✓ Restored persistent state into {PERSISTENT_DIR}")
    print(f"  Finished Stage 2d checkpoints found:  {restored_finals or 'none'}")
    print(f"  In-progress Stage 2d checkpoints found: {restored_steps or 'none'}")
    print(f"  Finished Stage 3 results found:       {restored_results or 'none'}")
else:
    print(
        "No previous session's Output is attached as an input yet -- "
        "starting fresh. After THIS run finishes and you commit it, attach "
        "this notebook's own Output as an input on the NEXT version "
        "(Add Input -> Notebook Output Files -> this notebook, latest "
        "version) so its checkpoints carry forward. See the documentation "
        "for the exact steps."
    )

↺ Found checkpoint state from a previous session: /kaggle/input/notebooks/parthbramhecha77/devnagri-restarte/persistent
✓ Restored persistent state into /kaggle/working/persistent
  Finished Stage 2d checkpoints found:  ['hindi']
  In-progress Stage 2d checkpoints found: ['hindi/step_2800']
  Finished Stage 3 results found:       ['hindi', 'marathi', 'sanskrit']


## Cross-session checkpoint restore

Kaggle wipes `/kaggle/working` between separate sessions -- only a **committed**
version's final `/kaggle/working` state is saved, as that version's "Output".
The cell below looks for a *previous* committed version of this notebook
attached as an input (see the documentation for the exact steps) and copies
its `persistent/` checkpoint tree back in before anything else runs, so
Stage 2d resumes instead of restarting from step 0.

In [4]:
# ============================================================
# 0b. RESTORE CHECKPOINT STATE FROM A PREVIOUS SESSION (IF ATTACHED)
# ============================================================
import shutil
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

def find_prior_persistent_dir():
    """Locate a 'persistent' directory inside any attached input dataset.

    This is how a PREVIOUS committed version of THIS SAME notebook's Output
    gets found, once it is attached as an input to the current version:
    Notebook -> Add Input -> Notebook Output Files -> this notebook,
    latest version. Kaggle mounts it read-only under
    /kaggle/input/<notebook-slug>/...
    """
    if not INPUT_ROOT.exists():
        return None
    matches = sorted(INPUT_ROOT.glob("*/persistent"))
    return matches[0] if matches else None

prior_persistent = find_prior_persistent_dir()

if prior_persistent is not None:
    print(f"↺ Found checkpoint state from a previous session: {prior_persistent}")
    PERSISTENT_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copytree(prior_persistent, PERSISTENT_DIR, dirs_exist_ok=True)

    restored_finals = sorted(
        p.parent.name for p in PERSISTENT_DIR.glob("models/devaware_finetuned/*/final")
    )
    restored_results = sorted(
        p.parent.name for p in PERSISTENT_DIR.glob("results/*/compression_results.json")
    )
    print(f"✓ Restored persistent state into {PERSISTENT_DIR}")
    print(f"  Finished Stage 2d checkpoints found: {restored_finals or 'none'}")
    print(f"  Finished Stage 3 results found:      {restored_results or 'none'}")
else:
    print(
        "No previous session's Output is attached as an input yet -- "
        "starting fresh. After THIS run finishes and you commit it, attach "
        "this notebook's own Output as an input on the NEXT version "
        "(Add Input -> Notebook Output Files -> this notebook, latest "
        "version) so its checkpoints carry forward. See the documentation "
        "for the exact steps."
    )


No previous session's Output is attached as an input yet -- starting fresh. After THIS run finishes and you commit it, attach this notebook's own Output as an input on the NEXT version (Add Input -> Notebook Output Files -> this notebook, latest version) so its checkpoints carry forward. See the documentation for the exact steps.


In [5]:
# ============================================================
# 1. REPOSITORY + DATASET — NON-DESTRUCTIVE
# ============================================================
import subprocess
import shutil

REPO_URL = "https://github.com/PARTH-BRAMHECHA/Devnagri_LLM.git"

# Never remove an existing repository. This preserves checkpoints and edits.
if REPO_DIR.exists():
    print("✓ Repository path already exists; reusing it.")
else:
    print("Repository not found; attempting clone...")
    result = subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        text=True,
    )
    if result.returncode != 0 or not REPO_DIR.exists():
        raise RuntimeError(
            "Could not clone Devnagri_LLM. "
            "Enable Kaggle Internet or attach/upload the repository."
        )

os.chdir(REPO_DIR)

git = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"],
    capture_output=True, text=True
)
print("✓ Repository:", REPO_DIR)
if git.returncode == 0:
    print("✓ Commit:", git.stdout.strip())

# Dataset link: create only if absent. Never delete a real directory.
data_dir = REPO_DIR / "data"
data_dir.mkdir(parents=True, exist_ok=True)
split_link = data_dir / "splits"

if split_link.is_symlink():
    if split_link.resolve() != SPLIT_SOURCE.resolve():
        split_link.unlink()
        split_link.symlink_to(SPLIT_SOURCE, target_is_directory=True)
elif split_link.exists():
    print("✓ data/splits already exists; leaving it untouched.")
else:
    if not SPLIT_SOURCE.exists():
        raise FileNotFoundError(
            f"Kaggle dataset was not found at {SPLIT_SOURCE}. "
            "Attach the split-devnagri-compression dataset."
        )
    split_link.symlink_to(SPLIT_SOURCE, target_is_directory=True)

print("Dataset path:", split_link)
print("Dataset resolves to:", split_link.resolve())


Repository not found; attempting clone...


Cloning into '/kaggle/working/Devnagri_LLM'...


✓ Repository: /kaggle/working/Devnagri_LLM
✓ Commit: a8a5ab6
Dataset path: /kaggle/working/Devnagri_LLM/data/splits
Dataset resolves to: /kaggle/input/datasets/parthbramhecha77/split-devnagri-compression/splits


In [6]:
# ============================================================
# 2. PERSISTENT OUTPUT DIRECTORIES — NON-DESTRUCTIVE
# ============================================================
import shutil

persistent_names = ["tokenizer", "models", "results", "logs", "eval"]

for name in persistent_names:
    target = PERSISTENT_DIR / name
    target.mkdir(parents=True, exist_ok=True)

    repo_path = REPO_DIR / name

    if repo_path.is_symlink():
        # Keep an existing symlink if it already resolves correctly.
        if repo_path.resolve() == target.resolve():
            continue
        repo_path.unlink()
    elif repo_path.exists():
        # A real directory already sits here (e.g. an empty folder shipped
        # in the repo, like "results/"). Merge anything it has into
        # persistent storage instead of leaving it as a real directory --
        # otherwise everything later written to it (Stage 3/4 results!)
        # lives only inside REPO_DIR and is lost the moment the repo is
        # re-cloned in the next Kaggle session, since REPO_DIR itself does
        # not survive across sessions the way PERSISTENT_DIR does.
        for item in repo_path.iterdir():
            dest = target / item.name
            if not dest.exists():
                shutil.move(str(item), str(dest))
        shutil.rmtree(repo_path)

    repo_path.symlink_to(target, target_is_directory=True)

print("✓ Persistent storage is ready:", PERSISTENT_DIR)
for name in persistent_names:
    p = REPO_DIR / name
    print(f"  {name:10s}: {p} -> {p.resolve() if p.exists() else 'missing'}")


✓ Persistent storage is ready: /kaggle/working/persistent
  tokenizer : /kaggle/working/Devnagri_LLM/tokenizer -> /kaggle/working/persistent/tokenizer
  models    : /kaggle/working/Devnagri_LLM/models -> /kaggle/working/persistent/models
  results   : /kaggle/working/Devnagri_LLM/results -> /kaggle/working/persistent/results
  logs      : /kaggle/working/Devnagri_LLM/logs -> /kaggle/working/persistent/logs
  eval      : /kaggle/working/Devnagri_LLM/eval -> /kaggle/working/persistent/eval


In [7]:
# ============================================================
# 3. VERIFY DATA + PROJECT FILES BEFORE INSTALLING
# ============================================================
required_files = [
    REPO_DIR / "run_pipeline.py",
    REPO_DIR / "requirements.txt",
]

for p in required_files:
    print(("✓" if p.exists() else "✗"), p)

if not all(p.exists() for p in required_files):
    raise FileNotFoundError("Required project files are missing.")

# Verify train.txt/test.txt exist for every language in LANGUAGES.
lang_data = {}
missing_lang_files = []

print("\nPer-language data:")
for lang in LANGUAGES:
    train_path = split_link / lang / "train.txt"
    test_path = split_link / lang / "test.txt"
    lang_data[lang] = {"train": train_path, "test": test_path}

    print(f"  {lang}:")
    print(f"    {'✓' if train_path.exists() else '✗'} {train_path}")
    print(f"    {'✓' if test_path.exists() else '✗'} {test_path}")

    if not train_path.exists() or not test_path.exists():
        missing_lang_files.append(lang)

if missing_lang_files:
    raise FileNotFoundError(
        "train.txt/test.txt are missing for: " + ", ".join(missing_lang_files) +
        ". Required language(s) must be present in the attached dataset."
    )

print("\n✓ Project and dataset verification passed for:", ", ".join(LANGUAGES))


✓ /kaggle/working/Devnagri_LLM/run_pipeline.py
✓ /kaggle/working/Devnagri_LLM/requirements.txt

Per-language data:
  hindi:
    ✓ /kaggle/working/Devnagri_LLM/data/splits/hindi/train.txt
    ✓ /kaggle/working/Devnagri_LLM/data/splits/hindi/test.txt

✓ Project and dataset verification passed for: hindi


In [ ]:
# ============================================================
# 4. REPAIR THE SCIENTIFIC PYTHON STACK ONCE
# ============================================================
# NOTE: Do NOT force-downgrade NumPy on Kaggle. The base image ships
# NumPy 2.0.2 and a dozen+ other pre-installed packages (jax, jaxlib,
# cupy-cuda12x, opencv-python, rasterio, pytensor, tifffile, shap,
# kaggle-environments, cesium, tobler, ...) require numpy>=2.0.
# Force-installing numpy==1.26.4 on top of that leaves NumPy in a
# corrupted/inconsistent state (this is what caused
# "ModuleNotFoundError: No module named 'numpy.char'" / 'numpy.strings').
#
# scipy==1.13.1 and scikit-learn==1.5.2 both support NumPy 2.x, so we
# pin THOSE to a consistent pair and reinstall them against whatever
# NumPy is already installed, instead of downgrading NumPy itself.

import subprocess
import sys
import numpy as np
from pathlib import Path

CURRENT_NUMPY = np.__version__
print(f"Detected existing NumPy: {CURRENT_NUMPY} (keeping this version)")

CONSTRAINTS = Path("/kaggle/working/devnagri_kaggle_constraints.txt")
CONSTRAINTS.write_text(
    "\n".join([
        f"numpy=={CURRENT_NUMPY}",
        "scipy==1.13.1",
        "scikit-learn==1.5.2",
        "transformers==4.50.3",
        "tokenizers==0.21.4",
        "huggingface_hub==0.36.2",
        "peft==0.15.2",
        "accelerate==1.14.0",
        "safetensors==0.8.0",
        "sentencepiece==0.2.2",
        "datasets==5.0.1",
        "bitsandbytes==0.50.1",
    ]) + "\n",
    encoding="utf-8",
)

print(CONSTRAINTS.read_text())

# Reinstall scipy/scikit-learn against the CURRENT numpy (no downgrade).
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "--no-cache-dir", "--force-reinstall",
    "-c", str(CONSTRAINTS),
    f"numpy=={CURRENT_NUMPY}",
    "scipy==1.13.1",
    "scikit-learn==1.5.2",
], check=True)

print("✓ NumPy/SciPy/scikit-learn stack repaired (NumPy version preserved).")


In [9]:
# ============================================================
# 5. INSTALL PROJECT + HUGGING FACE DEPENDENCIES ONCE
# ============================================================

req_path = REPO_DIR / "requirements.txt"

requirements_text = req_path.read_text(encoding="utf-8")

# ------------------------------------------------------------
# Synchronize repository pins with the Kaggle-controlled stack.
#
# The repository requirements.txt is NOT modified.
# A temporary Kaggle-specific requirements file is generated.
# ------------------------------------------------------------

REPLACEMENTS = {
    # WikiExtractor
    "wikiextractor==3.0.7": "wikiextractor==3.0.8",

    # Hugging Face / training stack
    "accelerate==1.1.1": "accelerate==1.14.0",
    "accelerate==1.1.0": "accelerate==1.14.0",

    "peft==0.15.0": "peft==0.15.2",
    "peft==0.15.1": "peft==0.15.2",

    "bitsandbytes==0.48.1": "bitsandbytes==0.50.1",
    "bitsandbytes==0.48.0": "bitsandbytes==0.50.1",

    "transformers==4.50.0": "transformers==4.50.3",
    "transformers==4.50.1": "transformers==4.50.3",
    "transformers==4.50.2": "transformers==4.50.3",

    "huggingface_hub==0.26.2": "huggingface_hub==0.36.2",
    "huggingface_hub==0.29.0": "huggingface_hub==0.36.2",
    "huggingface_hub==0.30.0": "huggingface_hub==0.36.2",

    "tokenizers==0.20.3": "tokenizers==0.21.4",
    "tokenizers==0.20.2": "tokenizers==0.21.4",

    "datasets==3.6.0": "datasets==5.0.1",

    "safetensors==0.5.3": "safetensors==0.8.0",
    "safetensors==0.5.2": "safetensors==0.8.0",

    "sentencepiece==0.2.0": "sentencepiece==0.2.2",
}

for old, new in REPLACEMENTS.items():
    if old in requirements_text:
        print(f"  Replacing: {old} -> {new}")
        requirements_text = requirements_text.replace(old, new)


TEMP_REQ = Path(
    "/kaggle/working/devnagri_requirements_kaggle.txt"
)

TEMP_REQ.write_text(
    requirements_text,
    encoding="utf-8",
)

print()
print("Temporary requirements:")
print("=" * 70)
print(TEMP_REQ.read_text(encoding="utf-8"))
print("=" * 70)


# ------------------------------------------------------------
# INSTALL PROJECT REQUIREMENTS
# ------------------------------------------------------------

subprocess.run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-cache-dir",
    "-c",
    str(CONSTRAINTS),
    "-r",
    str(TEMP_REQ),
], check=True)

print("✓ Project requirements installed.")


# ------------------------------------------------------------
# RE-ASSERT CRITICAL VERSIONS
#
# Do NOT reinstall NumPy/SciPy here.
# They were already repaired in Step 4.
# ------------------------------------------------------------

subprocess.run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-cache-dir",
    "-c",
    str(CONSTRAINTS),

    "transformers==4.50.3",
    "tokenizers==0.21.4",
    "peft==0.15.2",
    "accelerate==1.14.0",
    "bitsandbytes==0.50.1",
    "datasets==5.0.1",
    "huggingface_hub==0.36.2",
    "safetensors==0.8.0",
    "sentencepiece==0.2.2",
], check=True)

print("✓ Project and Hugging Face dependencies installed.")

  Replacing: accelerate==1.1.1 -> accelerate==1.14.0
  Replacing: bitsandbytes==0.48.1 -> bitsandbytes==0.50.1
  Replacing: huggingface_hub==0.26.2 -> huggingface_hub==0.36.2
  Replacing: tokenizers==0.20.3 -> tokenizers==0.21.4
  Replacing: sentencepiece==0.2.0 -> sentencepiece==0.2.2

Temporary requirements:
# ── Devnagri_LLM pinned dependencies ────────────────────────────────────────
# Installed via: pip install -r requirements.txt
# Pinned so Kaggle's preinstalled defaults never get picked up and silently
# conflict with what the pipeline code expects (e.g. torchao 0.10.0 default
# vs peft's LoRA dispatcher requiring torchao>=0.16.0).
#
# torch/torchvision/torchaudio are intentionally NOT pinned here — Kaggle's
# base image ships a CUDA-matched torch build, and reinstalling a different
# torch version from PyPI risks losing GPU support entirely.

# --- fixes the Stage 3 crash (PeftModel.from_pretrained / torchao check) ---
torchao==0.16.0
peft==0.15.2

# --- core HF / training sta

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tpot 1.1.0 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
a2a-sdk 0.3.26 requires protobuf>=5.29.5, but you have protobuf 5.29.3 which is incompatible.
category-encoders 2.9.0 requires scikit-learn>=1.6.0, but you have scikit-learn 1.5.2 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but 

✓ Project requirements installed.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 116.6 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 5.0.0
    Uninstalling datasets-5.0.0:
      Successfully uninstalled datasets-5.0.0
✓ Project and Hugging Face dependencies installed.


In [10]:
# ============================================================
# 6. IMPORT SMOKE TEST — FRESH KAGGLE EXECUTION
# ============================================================
# This cell was previously commented out entirely, which is why the
# NumPy corruption from Step 4 wasn't caught until 5 cells later
# (inside the checkpoint-repair loop). Keep this ACTIVE — it fails
# fast, right after the repair step, with a clear error instead of a
# confusing crash somewhere downstream.

import sys
import numpy as np
import scipy
import sklearn
import torch

print("=" * 72)
print("DEVNAGRI_LLM — IMPORT SMOKE TEST")
print("=" * 72)

print("\nPYTHON")
print("-" * 72)
print("Python:", sys.version)
print("Executable:", sys.executable)

print("\nSCIENTIFIC STACK")
print("-" * 72)
print("NumPy:", np.__version__)
print("NumPy path:", np.__file__)
print("SciPy:", scipy.__version__)
print("SciPy path:", scipy.__file__)
print("scikit-learn:", sklearn.__version__)
print("sklearn path:", sklearn.__file__)
print("PyTorch:", torch.__version__)

# ------------------------------------------------------------
# NumPy/SciPy compatibility tests (this is what would have caught
# the numpy.char bug immediately instead of 5 cells later)
# ------------------------------------------------------------

assert scipy.__version__ == "1.13.1", (
    f"Wrong SciPy loaded: {scipy.__version__}"
)

assert sklearn.__version__ == "1.5.2", (
    f"Wrong scikit-learn loaded: {sklearn.__version__}"
)

import numpy.char
print("✓ numpy.char import passed.")

import numpy.rec
print("✓ numpy.rec import passed.")

from scipy.special import sph_legendre_p
print("✓ scipy.special.sph_legendre_p import passed.")

from sklearn.metrics import roc_curve
print("✓ sklearn.metrics import passed.")


# ------------------------------------------------------------
# PyTorch / CUDA
# ------------------------------------------------------------

print("\nPYTORCH / CUDA")
print("-" * 72)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

assert torch.cuda.is_available(), "CUDA is not available."

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

x = torch.tensor([1.0, 2.0, 3.0], device="cuda")
print("✓ CUDA tensor allocation passed.")


# ------------------------------------------------------------
# Hugging Face — this import chain is exactly what crashed before,
# so if NumPy is broken, it will fail HERE instead of 5 cells later.
# ------------------------------------------------------------

import transformers
import huggingface_hub
import peft
import accelerate
import bitsandbytes
import tokenizers
import safetensors
import datasets
import sentencepiece

print("\nHUGGING FACE / TRAINING STACK")
print("-" * 72)

print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("Datasets:", datasets.__version__)
print("Safetensors:", safetensors.__version__)
print("SentencePiece:", sentencepiece.__version__)

# Prove the exact failing import from the traceback now works:
from transformers import AutoTokenizer
print("✓ from transformers import AutoTokenizer passed.")


# ------------------------------------------------------------
# Exact version checks (Hugging Face / training stack only —
# NumPy is intentionally NOT pinned to a fixed value here since we
# keep whatever NumPy the Kaggle image ships with)
# ------------------------------------------------------------

EXPECTED = {
    "transformers": ("4.50.3", transformers.__version__),
    "tokenizers": ("0.21.4", tokenizers.__version__),
    "huggingface_hub": ("0.36.2", huggingface_hub.__version__),
    "peft": ("0.15.2", peft.__version__),
    "accelerate": ("1.14.0", accelerate.__version__),
    "bitsandbytes": ("0.50.1", bitsandbytes.__version__),
    "datasets": ("5.0.1", datasets.__version__),
    "safetensors": ("0.8.0", safetensors.__version__),
    "sentencepiece": ("0.2.2", sentencepiece.__version__),
}

print("\nVERSION VERIFICATION")
print("-" * 72)

for name, (expected, actual) in EXPECTED.items():
    assert actual == expected, (
        f"{name}: expected {expected}, got {actual}"
    )
    print(f"✓ {name}: {actual}")


print("\n" + "=" * 72)
print("✓ ALL IMPORT SMOKE TESTS PASSED")
print("✓ SCIENTIFIC STACK PASSED")
print("✓ PYTORCH / CUDA PASSED")
print("✓ HUGGING FACE STACK PASSED")
print("=" * 72)
print("SAFE TO PROCEED TO STAGE 2d")
print("=" * 72)


ModuleNotFoundError: No module named 'numpy.strings'

In [ ]:
# ============================================================
# 7. GPU + HUGGING FACE AUTHENTICATION
# ============================================================

import os
import torch

# ------------------------------------------------------------
# 1. GPU CHECK
# ------------------------------------------------------------

print("=" * 60)
print("GPU / CUDA CHECK")
print("=" * 60)

print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable.\n"
        "Go to Kaggle Notebook → Settings → Accelerator → GPU."
    )

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)

    print(
        f"GPU {i}: {props.name} | "
        f"{props.total_memory / 1024**3:.1f} GiB"
    )

print("✓ CUDA/GPU check passed.")


# ------------------------------------------------------------
# 2. LOAD HUGGING FACE TOKEN FROM KAGGLE SECRETS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("HUGGING FACE AUTHENTICATION")
print("=" * 60)

HF_TOKEN = None

# Kaggle's official Secrets API
try:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()

    HF_TOKEN = secrets.get_secret("HF_TOKEN")

    if HF_TOKEN:
        print("✓ HF_TOKEN loaded from Kaggle Secrets.")
    else:
        print("⚠ Kaggle returned an empty HF_TOKEN.")

except Exception as e:
    print("⚠ Could not read HF_TOKEN from Kaggle Secrets.")
    print("   Error type:", type(e).__name__)

    # --------------------------------------------------------
    # Fallback: environment variable
    # --------------------------------------------------------
    HF_TOKEN = os.environ.get("HF_TOKEN")

    if HF_TOKEN:
        print("✓ HF_TOKEN loaded from environment variable.")


# ------------------------------------------------------------
# 3. VALIDATE TOKEN
# ------------------------------------------------------------

if not HF_TOKEN:
    raise RuntimeError(
        "\n"
        "HF_TOKEN is missing.\n\n"
        "Fix this in Kaggle:\n"
        "1. Open the Notebook.\n"
        "2. Go to Add-ons → Secrets.\n"
        "3. Add a secret named exactly:\n\n"
        "       HF_TOKEN\n\n"
        "4. Paste your Hugging Face token as the value.\n"
        "5. Enable the secret for this notebook.\n"
        "6. Restart/re-run this cell.\n"
    )

# Remove accidental whitespace/newlines
HF_TOKEN = HF_TOKEN.strip()

if not HF_TOKEN.startswith("hf_"):
    raise RuntimeError(
        "HF_TOKEN was found, but it does not look like a "
        "Hugging Face token. Make sure the secret contains "
        "your Hugging Face token beginning with 'hf_'."
    )

# Put token into environment for libraries that automatically
# look for HF_TOKEN.
os.environ["HF_TOKEN"] = HF_TOKEN

print("✓ HF_TOKEN is present.")
print("✓ Token format looks valid.")
print("✓ Token value hidden.")


# ------------------------------------------------------------
# 4. HUGGING FACE LOGIN / IDENTITY CHECK
# ------------------------------------------------------------

print("\nChecking Hugging Face authentication...")

from huggingface_hub import whoami

try:
    user = whoami(token=HF_TOKEN)

    username = (
        user.get("name")
        or user.get("fullname")
        or user.get("email")
        or "<unknown>"
    )

    print("✓ Hugging Face authentication successful.")
    print("Hugging Face user:", username)

except Exception as e:
    raise RuntimeError(
        "\n"
        "Hugging Face authentication failed.\n\n"
        "Possible causes:\n"
        "- HF_TOKEN is invalid or expired.\n"
        "- The token was copied incorrectly.\n"
        "- The token does not have the required permissions.\n"
        "- Kaggle is using an old/stale secret.\n\n"
        f"Original error: {type(e).__name__}: {e}"
    ) from e


# ------------------------------------------------------------
# 5. CHECK AIRAVATA MODEL ACCESS
# ------------------------------------------------------------

print("\nChecking Airavata model access...")

from huggingface_hub import model_info

AIRAVATA_MODEL = "ai4bharat/Airavata"

try:
    info = model_info(
        AIRAVATA_MODEL,
        token=HF_TOKEN
    )

    print("✓ Airavata model is accessible.")
    print("Model:", info.id)

    if getattr(info, "private", None) is not None:
        print("Private:", info.private)

    if getattr(info, "gated", None) is not None:
        print("Gated:", info.gated)

except Exception as e:
    raise RuntimeError(
        "\n"
        "Hugging Face authentication succeeded, but Airavata "
        "model access failed.\n\n"
        f"Model: {AIRAVATA_MODEL}\n"
        f"Error type: {type(e).__name__}\n"
        f"Error: {e}\n\n"
        "If the repository is gated, make sure your Hugging Face "
        "account has requested/received access to the model."
    ) from e


# ------------------------------------------------------------
# 6. FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("✓ ALL CHECKS PASSED")
print("=" * 60)

print("CUDA:              OK")
print("GPU(s):             OK")
print("HF_TOKEN:           OK")
print("HF authentication:  OK")
print("Airavata access:    OK")
print("=" * 60)

In [ ]:
from pathlib import Path
ckpt_root = PERSISTENT_DIR / "models" / "devaware_finetuned" / "hindi"
print("Exists:", ckpt_root.exists())
if ckpt_root.exists():
    for p in sorted(ckpt_root.iterdir()):
        print(" ", p.name, "-> files:", len(list(p.rglob("*"))) if p.is_dir() else "n/a")

In [ ]:
from pathlib import Path

print("prior_persistent:", prior_persistent)
print()
print("Contents of restored PERSISTENT_DIR:")
for p in sorted(PERSISTENT_DIR.rglob("*")):
    if p.is_dir():
        print(" DIR ", p.relative_to(PERSISTENT_DIR))

print()
print("Raw input mount, for comparison:")
for p in sorted(Path("/kaggle/input").rglob("*")):
    if p.is_dir():
        print(" IN  ", p)

## Stage 2 prerequisite

Stage 2d extends/fine-tunes the tokenizer/model state created by the earlier Stage 2 pipeline. If Stage 2 artifacts are already present, this cell can be skipped; otherwise run it once.

Runs once for **all three languages** via the repository's built-in `--lang all` mode. The notebook never deletes an existing checkpoint, so rerunning this later is safe and will resume/reuse what's already there.


In [ ]:
# ============================================================
# 8. STAGE 2 — PREREQUISITE (HINDI ONLY)
# ============================================================
import subprocess
import sys

os.chdir(REPO_DIR)

print("Running Stage 2 prerequisite for:", ", ".join(LANGUAGES))
ret = subprocess.call([
    sys.executable,
    "run_pipeline.py",
    "--stage", "2",
    "--lang", "hindi",
])

if ret != 0:
    raise RuntimeError(f"Stage 2 exited with code {ret}.")

print("✓ Stage 2 prerequisite completed for hindi.")


In [ ]:
# ============================================================
# 8b. REPAIR CORRUPTED tokenizer_class IN RESUMED CHECKPOINTS
# ============================================================
import json
import shutil
from pathlib import Path
from transformers import AutoTokenizer

def repair_tokenizer_class(ckpt_dir: Path):
    cfg_path = ckpt_dir / "tokenizer_config.json"
    if not cfg_path.exists():
        return

    try:
        AutoTokenizer.from_pretrained(ckpt_dir, trust_remote_code=True)
        return  # already loads fine, nothing to do
    except Exception as e:
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        current = cfg.get("tokenizer_class")
        print(f"  ⚠ {ckpt_dir}: tokenizer_class={current!r} fails to load ({e})")

    has_fast_file = (ckpt_dir / "tokenizer.json").exists()
    new_class = "PreTrainedTokenizerFast" if has_fast_file else "LlamaTokenizer"

    backup_path = cfg_path.with_suffix(".json.bak")
    if not backup_path.exists():
        shutil.copy2(cfg_path, backup_path)

    cfg["tokenizer_class"] = new_class
    with open(cfg_path, "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=2, ensure_ascii=False)

    AutoTokenizer.from_pretrained(ckpt_dir, trust_remote_code=True)  # confirm it now works
    print(f"  ✓ {ckpt_dir}: tokenizer_class {current!r} -> {new_class!r} (backup: {backup_path.name})")

checked = 0
for lang_dir in (PERSISTENT_DIR / "models" / "devaware_finetuned").glob("*"):
    if not lang_dir.is_dir():
        continue
    for ckpt_dir in list(lang_dir.glob("step_*")) + [lang_dir / "final"]:
        if ckpt_dir.exists():
            checked += 1
            repair_tokenizer_class(ckpt_dir)

print(f"\nChecked {checked} checkpoint director{'y' if checked == 1 else 'ies'}.")

# Stage 2d — vocabulary extension / fine-tuning (all languages)

This stage is resumable and runs **per language**, since each language has its own wall-clock training budget and its own checkpoint under `persistent/models/devaware_finetuned/<lang>`.

The notebook loops over `hindi`, `marathi`, and `sanskrit` in turn. For each language it keeps calling Stage 2d in bounded-time rounds until that language's `final` checkpoint appears (or the round budget for that language is exhausted, in which case rerunning this cell later resumes exactly where it left off — no checkpoint is ever deleted or reset).


In [ ]:
# ============================================================
# 9. STAGE 2d — HINDI ONLY, RESUMABLE
# ============================================================
import subprocess
import sys
import time
from pathlib import Path

os.chdir(REPO_DIR)

HOURS_PER_ROUND = 3.0
MAX_ROUNDS = 6

# Extra minutes each round costs beyond its own --max-hours budget:
# subprocess start, base-model + tokenizer reload, checkpoint restore,
# checkpoint save/prune on exit. Observed ~4-5 min/round in practice --
# pad generously so the check below fails safe.
ROUND_OVERHEAD_HOURS = 0.25

FINAL_DIRS = {
    lang: PERSISTENT_DIR / "models" / "devaware_finetuned" / lang / "final"
    for lang in LANGUAGES
}

for lang in LANGUAGES:
    final_dir = FINAL_DIRS[lang]

    if final_dir.exists():
        print(f"\n✓ {lang}: Stage 2d final checkpoint already exists, skipping.")
        print(f"  {final_dir}")
        continue

    print("\n" + "#" * 72)
    print(f"# STAGE 2d — {lang.upper()}")
    print("#" * 72)

    round_num = 0
    stopped_for_session_budget = False

    while not final_dir.exists() and round_num < MAX_ROUNDS:
        remaining = hours_remaining_in_session()
        needed = HOURS_PER_ROUND + ROUND_OVERHEAD_HOURS

        # This is the check that was MISSING: the old loop only ever asked
        # "do we have rounds left?" (round_num < MAX_ROUNDS). It never asked
        # "do we have SESSION time left?" so it kept launching 3h rounds
        # until Kaggle's 12h hard cap killed the kernel mid-Stage-3, with
        # no exception raised and no results written.
        if remaining < needed:
            print(
                f"\n⏱ Stopping Stage 2d for {lang} after round {round_num}: "
                f"only {remaining:.2f}h left in this Kaggle session "
                f"({KAGGLE_SESSION_LIMIT_HOURS:.1f}h cap - "
                f"{SESSION_SAFETY_BUFFER_HOURS:.1f}h reserved for Stage 3/4), "
                f"but another round needs ~{needed:.2f}h. "
                "The last checkpoint is already saved under "
                f"{PERSISTENT_DIR / 'models' / 'devaware_finetuned' / lang}, "
                "so it's safe to stop here."
            )
            stopped_for_session_budget = True
            break

        round_num += 1

        print("\n" + "=" * 72)
        print(f"STAGE 2d — {lang} — round {round_num}/{MAX_ROUNDS}")
        print(f"Session time used so far: {hours_elapsed():.2f}h / "
              f"{KAGGLE_SESSION_LIMIT_HOURS:.1f}h cap "
              f"({remaining:.2f}h left before the Stage 3/4 buffer).")
        print("Existing checkpoints are preserved; repository resumes when available.")
        print("=" * 72)

        t0 = time.time()

        ret = subprocess.call([
            sys.executable,
            "-m", "pipeline.stage2d_vocab_extend",
            "--lang", lang,
            "--max-hours", str(HOURS_PER_ROUND),
        ])

        elapsed = (time.time() - t0) / 60
        print(f"Round {round_num} exit code: {ret} | elapsed: {elapsed:.1f} min")

        if ret != 0:
            raise RuntimeError(
                f"Stage 2d failed for {lang} in round {round_num} with code {ret}. "
                "Inspect the output above before retrying."
            )

    if not final_dir.exists():
        if stopped_for_session_budget:
            raise RuntimeError(
                f"Stage 2d for {lang} did not finish within this Kaggle session's "
                f"time budget (stopped after round {round_num} with "
                f"{hours_remaining_in_session():.2f}h left). Its checkpoint was "
                "saved, so simply re-run this notebook in a fresh Kaggle session "
                "to resume Stage 2d from where it left off -- do NOT continue on "
                "to Stage 3 in this session."
            )
        raise RuntimeError(
            f"Stage 2d did not create {final_dir} for {lang} after "
            f"{MAX_ROUNDS * HOURS_PER_ROUND:.0f} hours of allowed rounds. "
            "Rerun this cell to continue from the saved checkpoint."
        )

    print(f"\n✓ {lang}: Stage 2d final checkpoint confirmed:")
    print(f"  {final_dir}")

print("\n" + "=" * 72)
print("✓ Stage 2d complete for:", ", ".join(LANGUAGES))
print(f"  Session time used: {hours_elapsed():.2f}h / {KAGGLE_SESSION_LIMIT_HOURS:.1f}h cap "
      f"({hours_remaining_in_session():.2f}h left before the Stage 3/4 buffer)")
print("=" * 72)


In [ ]:
# ============================================================
# 10. STAGE 2d GUARD — REQUIRED FOR STAGE 3
# ============================================================
from pathlib import Path

missing = []
for lang in LANGUAGES:
    final_dir = PERSISTENT_DIR / "models" / "devaware_finetuned" / lang / "final"
    status = "✓" if final_dir.exists() else "✗"
    print(f"{status} {lang}: {final_dir}")
    if not final_dir.exists():
        missing.append(lang)

if missing:
    raise RuntimeError(
        "Stage 2d final checkpoint is missing for: " + ", ".join(missing) +
        ". Stage 3's --use-devaware-tokenizer run is blocked until Stage 2d "
        "has produced a final checkpoint for hindi."
    )

# This is the second missing check from the original notebook: even once
# the checkpoint exists, Stage 3 (model load + three compression conditions)
# still needs real wall-clock time in THIS session. Without this check the
# notebook happily launched Stage 3's subprocess with ~5 minutes of session
# time left, which is exactly how the previous run died mid
# "Loading checkpoint shards" with no traceback.
remaining = hours_remaining_in_session()
if remaining < 0.5:
    raise RuntimeError(
        f"Only {remaining:.2f}h left in this Kaggle session "
        f"(elapsed {hours_elapsed():.2f}h / {KAGGLE_SESSION_LIMIT_HOURS:.1f}h cap). "
        "That's not enough headroom to safely start Stage 3 -- model loading "
        "alone can take several minutes, and a mid-load kill leaves no "
        "results file and no traceback to debug. Stage 2d's checkpoint is "
        "already saved, so re-run this notebook in a fresh Kaggle session and "
        "Stage 2d will skip straight to Stage 3."
    )

print("\n✓ Stage 2d final checkpoint exists for hindi.")
print(f"✓ {remaining:.2f}h left in session -- safe to run Stage 3 with --use-devaware-tokenizer.")


# Stage 3 — compression comparison (all languages)

Runs the three intended conditions for **hindi**, **marathi**, and **sanskrit** in a single call using `--lang all`:
1. classical compression;
2. pretrained LLM with its default tokenizer;
3. the DevAware / fine-tuned tokenizer from Stage 2d.

Using `--lang all` loads the 7B model once and reuses it across all three languages, rather than reloading it three times. The verification below fails loudly if any language is missing a required condition.


In [ ]:
# ============================================================
# 11. STAGE 3 — HINDI ONLY, THREE-CONDITION COMPRESSION
# ============================================================
import subprocess
import sys
import json
from pathlib import Path

os.chdir(REPO_DIR)

ret = subprocess.call([
    sys.executable,
    "run_pipeline.py",
    "--stage", "3",
    "--lang", "hindi",
    "--use-devaware-tokenizer",
])

if ret != 0:
    raise RuntimeError(f"Stage 3 exited with code {ret}.")

required_conditions = [
    "classical",
    "llm_compression",
    "llm_compression_devaware_tokenizer",
]

all_comp_results = {}
lang_missing = {}

for lang in LANGUAGES:
    comp_path = REPO_DIR / "results" / lang / "compression_results.json"

    if not comp_path.exists():
        raise RuntimeError(
            f"{comp_path} was not written. Stage 3 did not complete for {lang}."
        )

    comp_results = json.loads(comp_path.read_text(encoding="utf-8"))
    all_comp_results[lang] = comp_results

    missing = []
    for key in required_conditions:
        value = comp_results.get(key)
        if not value or (isinstance(value, dict) and "error" in value):
            missing.append(key)
    if missing:
        lang_missing[lang] = missing

if lang_missing:
    raise RuntimeError(
        "Stage 3 completed without all three required conditions for: " +
        "; ".join(f"{lang} (missing {', '.join(keys)})" for lang, keys in lang_missing.items())
    )

print("✓ Stage 3 results written for:", ", ".join(LANGUAGES))
print("✓ All three conditions are present.")

for lang in LANGUAGES:
    print(f"\n=== {lang} ===")
    comp_path = REPO_DIR / "results" / lang / "compression_results.json"
    print("  Results file:", comp_path)
    for key in required_conditions:
        item = all_comp_results[lang][key]
        if isinstance(item, dict):
            print(
                f"  {key}: "
                f"BPC={item.get('bpc', 'n/a')} | "
                f"ratio={item.get('compression_ratio', 'n/a')}"
            )


# Stage 4 — final comparison (all languages)

Stage 4 consumes each language's verified Stage 3 JSON and produces the final comparison/report, plus a cross-language master table (`generate_master_table()`, run automatically by `--lang all`).


In [ ]:
# ============================================================
# 12. STAGE 4 — HINDI ONLY, FINAL REPORT
# ============================================================
import subprocess
import sys
import json
from pathlib import Path

os.chdir(REPO_DIR)

for lang in LANGUAGES:
    comp_path = REPO_DIR / "results" / lang / "compression_results.json"
    if not comp_path.exists():
        raise RuntimeError(f"Stage 3 results do not exist for {lang}. Run Stage 3 first.")

ret = subprocess.call([
    sys.executable,
    "run_pipeline.py",
    "--stage", "4",
    "--lang", "hindi",
])

if ret != 0:
    raise RuntimeError(f"Stage 4 exited with code {ret}.")

print("\n✓ Stage 4 completed for hindi.")

for lang in LANGUAGES:
    print("\n" + "#" * 72)
    print(f"# {lang.upper()}")
    print("#" * 72)
    for filename in [
        "baseline_tokenizer_results.json",
        "compression_results.json",
        "devanagari_tokenizer_comparison.json",
    ]:
        path = REPO_DIR / "results" / lang / filename
        print(f"\n=== {filename} ===")
        if path.exists():
            data = json.loads(path.read_text(encoding="utf-8"))
            print(json.dumps(data, indent=2, ensure_ascii=False)[:5000])
        else:
            print("Not produced by this repository run.")


# Stage 5 — cross-lingual analysis (SKIPPED in this hindi-only notebook)

Cross-lingual analysis compares languages against each other, so it has
nothing to do with a single language. This cell is intentionally left out
of the hindi-only run. Re-add the original Stage 5 cell once marathi and
sanskrit also have their own Stage 3/4 results.

In [ ]:
# ============================================================
# 12b. STAGE 5 — CROSS-LINGUAL ANALYSIS (SKIPPED, HINDI-ONLY RUN)
# ============================================================
print("Skipping Stage 5 (cross-lingual analysis) — this notebook only "
      "processes hindi, and cross-lingual analysis requires results from "
      "multiple languages.")


In [ ]:
# ============================================================
# 13. FINAL SANITY SUMMARY
# ============================================================
import os
from pathlib import Path

print("=" * 72)
print("DEVNAGRI LLM PIPELINE — FINAL STATUS (HINDI ONLY)")
print("=" * 72)

checks = {"Repository": REPO_DIR.exists()}

for lang in LANGUAGES:
    checks[f"{lang.capitalize()} train"] = lang_data[lang]["train"].exists()
    checks[f"{lang.capitalize()} test"] = lang_data[lang]["test"].exists()
    checks[f"{lang.capitalize()} Stage 2d final"] = (
        PERSISTENT_DIR / "models" / "devaware_finetuned" / lang / "final"
    ).exists()
    checks[f"{lang.capitalize()} Stage 3 results"] = (
        REPO_DIR / "results" / lang / "compression_results.json"
    ).exists()
    checks[f"{lang.capitalize()} Stage 4 directory"] = (
        REPO_DIR / "results" / lang
    ).exists()

for name, ok in checks.items():
    print(f"{'✓' if ok else '✗'} {name}")

print("\nRepository:", REPO_DIR)
print("Persistent:", PERSISTENT_DIR)
print("Languages:", ", ".join(LANGUAGES))
for lang in LANGUAGES:
    print(f"Results ({lang}):", REPO_DIR / "results" / lang)

print(f"\nSession time used: {hours_elapsed():.2f}h / {KAGGLE_SESSION_LIMIT_HOURS:.1f}h cap")
print("\n✓ Notebook completed without destructive cleanup.")
print("Note: marathi/sanskrit were not processed in this hindi-only run.")

all_final = all(
    (PERSISTENT_DIR / "models" / "devaware_finetuned" / lang / "final").exists()
    for lang in LANGUAGES
)
all_results = all(
    (REPO_DIR / "results" / lang / "compression_results.json").exists()
    for lang in LANGUAGES
)

print("\n" + "=" * 72)
if all_final and all_results:
    print("✓ PIPELINE COMPLETE for:", ", ".join(LANGUAGES))
    print("  Click 'Save Version' now to commit this run.")
else:
    print("⚠ PIPELINE NOT YET COMPLETE -- click 'Save Version' to commit this")
    print("  run's checkpoint progress, THEN, before starting the next run:")
    print("  Notebook -> Add Input -> Notebook Output Files -> select this")
    print("  notebook's latest version, so its checkpoints carry forward.")
print("=" * 72)
